# Chapter 16 — Change the Model, Change the Universe

**Book alignment:** Embeddings From First Principles, Chapter 16

**Question this notebook isolates:** Two encoders embed the same text into unrelated
coordinate systems — even at equal dimension. What, if anything, is shared? On RELATE
(Wave 3), the axis of agreement is **shared training regime, not equal dimension**: two
1024-d retrieval-tuned models sit at linear CKA 0.99 / 88% neighbourhood overlap, while
cross-family pairs sit at ~0.82 / ~70% and disagree on a third of top-1 retrievals.

In [ ]:
from pathlib import Path
import json
import numpy as np

rng = np.random.default_rng(0)


def find_repo_root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / "experiments" / "embeddings-from-first-principles" / "wave1").is_dir():
            return c
    raise RuntimeError("run from a checkout containing experiments/embeddings-from-first-principles")


ROOT = find_repo_root(Path.cwd().resolve())
EXP = ROOT / "experiments" / "embeddings-from-first-principles"


def art(wave, name):
    sub = "wave1/artifacts/v02" if wave == "wave1-v02" else f"{wave}/artifacts"
    return json.loads((EXP / sub / name).read_text())

## 1. Same text, two encoders, orthogonal vectors — and why structural comparison is the only option

In [ ]:
n, d = 300, 64
truth = rng.standard_normal((n, d))                       # a shared underlying structure

# two "encoders": each applies its own random rotation + scaling + its own noise
def encoder(seed):
    r = np.random.default_rng(seed)
    W = r.standard_normal((d, d))
    return lambda X: (X @ W + r.standard_normal((len(X), d)) * 0.3)

EA, EB = encoder(1), encoder(2)
A, B = EA(truth), EB(truth)
A /= np.linalg.norm(A, axis=1, keepdims=True)
B /= np.linalg.norm(B, axis=1, keepdims=True)

same_text_cos = np.mean([float(A[i] @ B[i]) for i in range(n)])
print(f"cos(A[i], B[i]) for the SAME text, averaged: {same_text_cos:+.3f}   (~ orthogonal)")
assert abs(same_text_cos) < 0.2
print("no shared origin, axes, or scale - coordinate-wise comparison is undefined")

In [ ]:
def linear_cka(X, Y):
    X = X - X.mean(0); Y = Y - Y.mean(0)
    hsic = np.linalg.norm(Y.T @ X, "fro") ** 2
    return float(hsic / (np.linalg.norm(X.T @ X, "fro") * np.linalg.norm(Y.T @ Y, "fro")))

def nn_overlap(X, Y, k=10):
    def knn(M):
        return np.argsort(-(M @ M.T), axis=1)[:, 1:k + 1]
    a, b = knn(X), knn(Y)
    return float(np.mean([len(set(x) & set(y)) / k for x, y in zip(a, b)]))

print(f"linear CKA(A, B)        = {linear_cka(A, B):.3f}   (similarity up to a linear map: high)")
print(f"10-NN overlap(A, B)     = {nn_overlap(A, B):.3f}   (relational structure partly shared)")
assert linear_cka(A, B) > 0.5
print("compare STRUCTURE (who is near whom), never coordinates")

## 2. On RELATE, shared training regime — not dimension — predicts agreement (Wave 3)

In [ ]:
sc = art("wave3", "space-comparison.json")["pairs"]
for pair, v in sc.items():
    print(f"  {pair:30} CKA {v['linear_cka']:.3f}  10-NN {v['neighborhood_overlap_at10']:.3f}"
          f"  top1-agree {v['top1_retrieval_agreement']:.3f}   [{v['note']}]")

same_regime = sc["bge-large vs mxbai-large"]          # same dim, both retrieval-tuned, diff creators
cross_family = sc["mpnet-base vs bge-large"]          # different family
assert same_regime["linear_cka"] > 0.98 and same_regime["neighborhood_overlap_at10"] > 0.85
assert cross_family["linear_cka"] < 0.9 and cross_family["top1_retrieval_agreement"] < 0.75
print("\n'both are 768-dimensional' predicts nothing; 'both retrieval-tuned, same width' predicts a lot")
print("high CKA still != shared decisions: the cross-family pairs disagree on ~1/3 of top-1 retrievals")

## What we earned

Two models' spaces share no origin, axes, or scale — regardless of output dimension. You
compare *structure*: neighbourhood overlap, CKA, retrieval agreement. Coarse topical
structure is usually shared; fine, hard-negative, rare-item, and calibration structure
usually is not. On RELATE, two similarly-trained 1024-d models are nearly the same space
(CKA 0.99); cross-family pairs sit at CKA ~0.82 and disagree on a third of top-1 results.
**Equal dimensions do not imply compatible representation.**

**Notebook 17 / Chapter 17** treats a model *upgrade* as a model swap you did to yourself
over a corpus you already stored.